In [1]:
## Install the dependencies
%pip install -r requirements.txt

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


### Set paths and check hardware

In [ ]:
import csv
import json
import shutil
import subprocess
import unicodedata
from pathlib import Path

import jiwer
import numpy as np
import soundfile as sf
import torch
import transformers
from IPython.display import Audio, display
from torch.utils.data import Dataset 
from tqdm.auto import tqdm
from transformers import (Seq2SeqTrainer, Seq2SeqTrainingArguments, WhisperForConditionalGeneration, WhisperProcessor)

ROOT = Path.cwd().resolve()
if ROOT.name == "../../myanmar-asr":
    ROOT = ROOT.parent

assert (ROOT / "dataset" / "NLP Project1" / "audio_files").is_dir(), f"Wrong Jupyter working directory: {ROOT}"
assert (ROOT / "dataset" / "asr_work" / "train_whisper.csv").is_file(), f"Missing prepared manifests" 
assert shutil.which("ffmpeg"), "Installing FFmpeg and restart Jupyter so ffmpeg is on PATH" 

ModuleNotFoundError: No module named 'jiwer'

In [ ]:
QUICK_RUN = False
MODEL_ID = "openai/whisper-small" 
RUN_DIR = ROOT / "asr_work" / ("ucsy_whisper_smoke" if QUICK_RUN else "ucsy_whisper_full")
CACHE_DIR = ROOT / "asr_work" / "ucsy_audio_16k" 
RUN_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
device = "cuda" if torch.cuda.is_available() else "mps" if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available() else "cpu"
print({
    "root": str(ROOT),
    "device": device,
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "ffmpeg": shutil.which("ffmpeg")
})

if device == "cpu":
    print("CPU can check the data pipeline, but full Whisper needs a GPU in practice.")

NameError: name 'torch' is not defined

### Load UCSY-only splits and checking leakage

In [ ]:
def load_ucsy_split(name):
    with (ROOT / "asr_work" / f"{name}_whisper.csv").open(newline="", encoding="utf-8") as handle:
        rows = [row for row in csv.DictReader(handle) if row["source"] == "ucsy"]
    assert rows, f"No UCSY rows in {name}"

    for row in rows:
        path = (ROOT / row["audio"]).resolve()
        assert path.is_relative_to((ROOT / "NLP Project1" / "audio_files").resolve()), row["audio"]
        assert path.is_file() and path.stat().st_size > 0, row["audio"]
        assert 0 < float(row["duration_seconds"]) < 30, row["sample_id"]
    return rows

In [ ]:
# load all train, test and validation
all_train = load_ucsy_split("train")
all_test = load_ucsy_split("test_ucsy")
all_validation = load_ucsy_split("validation")

In [ ]:
for key in ("sha256", "record_group", "text"):
    sets = [set(row[key] for row in rows) for rows in (all_train, all_test, all_validation)]
    assert not (sets[0] & sets[1] or sets[0] & sets[2] or sets[1] & sets[2]), f"{key} cross split"

# for the dataset split summary
def summary(rows):
    return {
        "clips": len(rows),
        "hours": round(sum(float(r["duration_s"]) for r in rows) / 3600, 3),
        "questions": sum(r["side"] == "question" for r in rows),
        "answers": sum(r["side"] == "answer" for r in rows),
    }

print({
    "train": summary(all_train),
    "test": summary(all_test),
    "validation": summary(all_validation)
})

In [ ]:
rng = random.Random(20261030)
train_rows, validation_rows = all_train.copy(), all_validation.copy()

if QUICK_RUN:
    rng.shuffle(train_rows)
    rng.shuffle(validation_rows)
    train_rows = train_rows[:256]
    validation_rows = validation_rows[:64]

print("Selected: ", summary(train_rows), summary(validation_rows))

### Listen to two labels

In [ ]:
for side in ("question", "answer"):
    row = next(r for r in train_rows if r["side"] == side)
    print(side, row["sample_id"], row["text"])
    display(Audio(filename=str(ROOT / row["audio"])))

### Load Whisper and filter labels to its target length

In [ ]:
processor = WhisperProcessor.from_pretrained(MODEL_ID, language="myanmar", task="transcribe")
model = WhisperForConditionalGeneration.from_pretrained(MODEL_ID)
model.generation_config.language = "myanmar"
model.generation_config.task = "transcribe"
model.generation_config.forced_decoder_ids = None
model.config.use_cache = False # Required when using gradient checkpointing, 
model.generation_config.use_cache = True
MAX_LABEL_TOKENS = model.config.max_target_positions

def filter_label_length(rows):
    accepted, rejected = [], []
    for row in rows:
        length = len(processor.tokenizer(row["text"]).input_ids)
        (accepted if length <= MAX_LABEL_TOKENS else rejected).append(row)
    return accepted, rejected

train_rows, rejected_train = filter_label_length(train_rows)
validation_rows, rejected_validation = filter_label_length(validation_rows)
assert train_rows and validation_rows

with (RUN_DIR / "long_label.csv").open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(handle, fieldnames=["sample_id", "audio", "text"])
    writer.writeheader()
    writer.writerows({
            key: row[key] for key in writer.fieldnames}
            for row in rejected_train + rejected_validation
        )

print({
    "max_label_tokens": MAX_LABEL_TOKENS,
    "train": summary(train_rows),
    "validation": summary(validation_rows),
    "long_label": len(rejected_train) + len(rejected_validation)
})